In [1]:
import pyspark.sql.types as t
from gentropy.common.session import Session
from pyspark.sql import functions as f


Loading BokehJS ...

/Users/yt4/Projects/Gentropy-manuscript/.venv/lib/python3.11/site-packages/pyspark/sql/pandas/functions.py:407: UserWarning:

In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.



In [2]:
session = Session(extended_spark_conf={"spark.driver.memory": "10g"})


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/27 12:21:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
path_to_release_folder = "../../data/25.06/"

output_path = "../../data/intermediate_files/"


In [ ]:
# combinig it with l2g predictions
l2g = session.spark.read.parquet(path_to_release_folder + "irene_1208_l2g_predictions").select(
    "studyLocusId", "geneId", "score"
)


In [ ]:
fm = session.spark.read.parquet(path_to_release_folder + "../intermediate_files/l2g_feature_matrix")
fm = fm.filter(f.col("isProteinCoding") == 1).cache()
fm.count()


25/11/27 12:24:13 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


10623371

In [ ]:
combined_df = (
    fm.join(l2g.select("studyLocusId", "geneId", "score"), on=["geneId", "studyLocusId"], how="left").fillna(0).cache()
)
combined_df.count()


10623371

In [ ]:
clpp_thr = 0.01
coloc_thr = 0.8

combined_df = combined_df.withColumn(
    "eQTL_coloc",
    f.when((f.col("eQtlColocClppMaximum") >= clpp_thr) | (f.col("eQtlColocH4Maximum") >= coloc_thr), 1).otherwise(0),
)
combined_df = combined_df.withColumn(
    "pQTL_coloc",
    f.when((f.col("pQtlColocClppMaximum") >= clpp_thr) | (f.col("pQtlColocH4Maximum") >= coloc_thr), 1).otherwise(0),
)
combined_df = combined_df.withColumn("VEP", f.when((f.col("vepMaximum") >= 0.66), 1).otherwise(0))
# combined_df = combined_df.withColumn(
#    "distance",
#    f.when((f.col("distanceSentinelFootprintNeighbourhood")==1) |
#    (f.col("distanceSentinelTssNeighbourhood")==1), 1).otherwise(0)
# ).cache()

combined_df = combined_df.withColumn(
    "distanceTSS", f.when(f.col("distanceSentinelTssNeighbourhood") == 1, 1).otherwise(0)
).cache()


combined_df.count()


10623371

In [ ]:
l2g_05 = l2g.filter(f.col("score") >= 0.5).cache()
cs_with_l2g = l2g_05.select("studyLocusId").distinct()
cs_with_l2g.count()


480623

In [ ]:
l2g_05.count()


503497

In [ ]:
l2g_no_05 = l2g.join(cs_with_l2g, on="studyLocusId", how="left_anti").cache()
l2g_no_05.count()


688918

In [ ]:
l2g_no_05.select("studyLocusId").distinct().count()


292355

In [ ]:
from pyspark.sql import Window

# Define a window specification partitioned by studyLocusId and ordered by score descending
window_spec = Window.partitionBy("studyLocusId").orderBy(f.desc("score"))

# Add a row_number column to rank rows within each studyLocusId partition
l2g_no_05_ranked = l2g_no_05.withColumn("row_number", f.row_number().over(window_spec))

# Filter rows where row_number is 1 (i.e., the max score for each studyLocusId)
l2g_no_05_max = l2g_no_05_ranked.filter(f.col("row_number") == 1).drop("row_number")

# Show the result
l2g_no_05_max.count()


292355

In [ ]:
l2g_no_05_max.select("studyLocusId").distinct().count()


292355

In [ ]:
l2g_no_05_max.show(4)


+--------------------+---------------+-------------------+
|        studyLocusId|         geneId|              score|
+--------------------+---------------+-------------------+
|002462a2da2f7c279...|ENSG00000215547|0.36747118830680847|
|00a70f45252881a9f...|ENSG00000254636| 0.3146820664405823|
|00badf4cd2ff71a2c...|ENSG00000105655| 0.2848552167415619|
|00bbe340d79fe1aa1...|ENSG00000077549|  0.485416442155838|
+--------------------+---------------+-------------------+
only showing top 4 rows



In [ ]:
l2g_no_05_max_01 = l2g_no_05_max.filter(f.col("score") >= 0.1).cache()
l2g_no_05_max_01.count()


285270

In [ ]:
l2g_prioritised = l2g_no_05_max_01.unionByName(l2g_05.select("studyLocusId", "geneId", "score")).cache()
l2g_prioritised.count()


788767

In [ ]:
261020 + 524241


785261

In [ ]:
l2g_prioritised.show(1)


+--------------------+---------------+-------------------+
|        studyLocusId|         geneId|              score|
+--------------------+---------------+-------------------+
|002462a2da2f7c279...|ENSG00000215547|0.36747118830680847|
+--------------------+---------------+-------------------+
only showing top 1 row



In [ ]:
final = (
    combined_df.select("studyLocusId", "geneId", "score", "eQTL_coloc", "pQTL_coloc", "VEP", "distanceTSS")
    .join(l2g_prioritised.drop("score"), on=["studyLocusId", "geneId"], how="inner")
    .cache()
)
final.count()


788767

In [ ]:
final.show(2)


+--------------------+---------------+------------------+----------+----------+---+-----------+
|        studyLocusId|         geneId|             score|eQTL_coloc|pQTL_coloc|VEP|distanceTSS|
+--------------------+---------------+------------------+----------+----------+---+-----------+
|2ef4eb65d7f430a75...|ENSG00000000971|0.8191224336624146|         0|         0|  0|          1|
|469a4c2b6f1247cbe...|ENSG00000000971|0.6837719082832336|         1|         0|  1|          1|
+--------------------+---------------+------------------+----------+----------+---+-----------+
only showing top 2 rows



In [ ]:
final.select("studyLocusId").distinct().count()


765893

In [ ]:
final.write.mode("overwrite").parquet(output_path + "list_of_prioritised_genes_per_CS.parquet")
